# 01 — What a voice agent actually is

## The machine you are evaluating
A modern voice agent is three models in a relay race, run in a loop, under brutal time pressure:

1. **STT / ASR** (speech-to-text, a.k.a. automatic speech recognition) — turns the caller's audio into text, ideally *streaming* (words appear while the person is still talking). Quality metric: **WER**, word error rate.
2. **LLM** — decides what to say next given the conversation so far (this is where the "agent" lives: prompts, tools, business logic).
3. **TTS** (text-to-speech) — turns the reply into audio. Cartesia's whole company is this box. Their headline metric is **TTFA** — time to first audio: how many ms until the voice starts. Sonic 3.5 claims 82ms.

The relay happens **every turn**, so per-turn latency = STT finalization + LLM first tokens + TTS first audio + network. Humans hand off the floor in ~200–300ms, so anything the stack adds is felt immediately. That is why our rubric calls ≤300ms snappy and >800ms laggy — past ~800ms callers start saying "hello? are you there?"

## The hard part is not the models — it is turn-taking
The agent must decide **when the caller is done talking**. That decision is called **endpointing**, and it is built on **VAD** — voice activity detection: a tiny classifier answering "is there speech right now?" frame by frame. Endpointing is basically: VAD says silence + silence has lasted X ms → caller is done → respond.

Get X too small → the agent answers during the caller's thinking pause → **barge-in** (the agent interrupts a human). Get X too big → dead air after every caller turn → laggy. *Every voice agent lives on this knife edge.* Our hero call's two sins are exactly the two ways to fall off it. Not a coincidence — that is the demo's thesis.

Two more terms you will hear in the room:
- **Backchannel** — tiny overlapping listener noises ("mm-hmm", "haan") that do NOT claim the floor. Healthy conversation has them; that is why overlaps ≤100ms do not count as barge-ins in our rubric.
- **Diarization** — figuring out who-spoke-when from raw audio when you do not have separate channels. Error-prone, and a classic way demos die. We engineered around it twice: SpokenWOZ gives speaker tags, and the hero call's timestamps come from how we assembled it.

And one from our own data: **code-switching** — mixing languages mid-sentence ("Madhapur side anukunta, near the metro station"). Real Indian callers do this constantly; most English-trained stacks degrade on it. It is why `language` sits in our schema from day one.

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "rubric.yaml").exists())
import sys
sys.path.insert(0, str(ROOT / "pipeline"))
print("repo root:", ROOT)

import json, wave, numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio

w = wave.open(str(ROOT / "data" / "hero" / "hero_001.wav"))
sr, n = w.getframerate(), w.getnframes()
sig = np.frombuffer(w.readframes(n), dtype=np.int16).astype(np.float32) / 32768
call = json.loads((ROOT / "data" / "hero" / "turns.json").read_text())
print(f"{len(sig)/sr:.1f}s of audio · {len(call['turns'])} turns · sr={sr}")
Audio(str(ROOT / "data" / "hero" / "hero_001.wav"))

**PREDICT** before running the next cell: the plot will show one bar per turn (teal = you, purple = agent). Where will bars *overlap vertically in time*, and where will there be a visible horizontal hole? Say the timestamps out loud — you recorded this call.

In [ ]:
t = np.arange(len(sig)) / sr
fig, ax = plt.subplots(figsize=(13, 3.5))
ax.plot(t, sig, lw=0.3, color="#999", alpha=0.7)
for tn in call["turns"]:
    c = "#1D9E75" if tn["speaker"] == "user" else "#7F77DD"
    y = (0.75, 1.05) if tn["speaker"] == "user" else (-1.05, -0.75)
    ax.fill_betweenx(y, tn["start_ms"]/1000, tn["end_ms"]/1000, color=c, alpha=0.65)
    ax.text((tn["start_ms"]+tn["end_ms"])/2000, y[0]+0.12, tn["turn_id"], ha="center", fontsize=8)
ax.set(xlabel="seconds", yticks=[], title="hero_001 — caller (top) vs agent (bottom)")
plt.show()

Look at `t2`→`t3`: the purple bar starts while teal is still running — the **barge-in**, exactly 0.8s of double-talk. Look at `t6`→`t7`: a hole — **dead air**, 1.62s. You can *see* the two sins. The failure table is just this picture, written as numbers.

## Build a toy VAD in 6 lines
The booth's hands-free flow tonight was exactly this idea: chop audio into 30ms frames, compute energy (RMS), threshold it. **PREDICT:** during your t2 thinking pause, what will the mask show?

In [ ]:
frame = int(0.030 * sr)
trimmed = sig[: len(sig) // frame * frame]
rms = np.sqrt((trimmed.reshape(-1, frame) ** 2).mean(axis=1))
speech = rms > 0.02
tf = (np.arange(len(rms)) * frame + frame/2) / sr
fig, ax = plt.subplots(figsize=(13, 2))
ax.fill_between(tf, 0, speech.astype(int), step="mid", color="#1D9E75", alpha=0.7)
ax.set(xlabel="seconds", yticks=[0, 1], yticklabels=["silence", "speech"], title="energy VAD, 30ms frames")
plt.show()
print(f"speech fraction: {speech.mean():.0%}")

That dip inside t2 is your scripted pause — a VAD with a short endpointing hold would have cut you off there, which is why the booth waited 3.2s on that turn and why real agents barge in on hesitant callers. **An entire product failure class, visible in 6 lines of numpy.**

## Exercise — connect to the pipeline
Use the repo's own `turn_metrics` (the function the failure table is built on) and find both sins programmatically. Fill in the blank, predict the two numbers first.

In [ ]:
from signals import turn_metrics
events = turn_metrics(call["turns"])
sins = [e for e in events if e["overlap_ms"] > 100 or e["gap_ms"] > 800]   # YOUR TURN first: why these two conditions?
for e in sins:
    print(f"{e['prev_turn_id']}->{e['next_turn_id']}  fto={e['fto_ms']:+}ms  "
          f"overlap={e['overlap_ms']}  gap={e['gap_ms']}  ({e['prev_spk']}->{e['next_spk']})")

## Self-check (out loud, then open answers)
1. Name the three models in the relay and the metric each is judged by.
2. What is endpointing, and what are the two opposite failure modes of getting it wrong?
3. Why does our rubric ignore overlaps under 100ms?
4. Why did we go out of our way to avoid diarization in this project?
5. A founder says "our agent's average response time is 600ms, we're fine." What two follow-up questions does this notebook arm you to ask?

<details><summary>Answers</summary>

1. STT/ASR (WER), LLM (task quality — what our judge scores), TTS (TTFA / naturalness).
2. Deciding the caller is finished. Too eager → barge-in over thinking pauses; too patient → dead air / laggy responses.
3. Sub-100ms overlaps are mostly backchannels — cooperative listener noises that do not claim the floor. Counting them would flood the table with non-failures.
4. Who-spoke-when from raw audio is error-prone; bad speaker boundaries poison every downstream timing number. We got speaker truth for free instead (channel tags in SpokenWOZ, assembly in the hero call).
5. "What is the p90, not the mean?" and "is that measured per turn from caller-end-of-speech to first audio, or something flattering?" — tails and measurement definitions are where averages hide failure.
</details>